# causal/05 DAG 因果图 实战

数据：5-node 合成 DAG (Z=confounder, X=treatment, M=mediator, Y=outcome, W=collider)
目标：用真实样本验证 d-separation 定理 + backdoor 准则

三步法：
1. 偏相关矩阵：观察哪些变量条件独立 / 不独立
2. d-separation 三种结构验证：链/叉/对撞
3. Backdoor adjustment：naive 回归 vs 控制 {Z} 后回归

In [1]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'articles' / 'causal' / 'data'))
from load_causal_dag import load_causal_dag

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

data = load_causal_dag()
df = data['df']
print(f'5-node DAG with {len(data["edges"])} edges')
print(f'Sample size: {data["n_samples"]}')
print(f'Nodes: {data["nodes"]}')
print(f'Edges: {data["edges"]}')
print(f'True direct effect X -> Y: {data["coeffs"]["X->Y"]}')
print(f'True total effect X -> Y: {data["coeffs"]["X->Y"] + data["coeffs"]["X->M"] * data["coeffs"]["M->Y"]:.3f}')
print()
print('Head of dataset:')
print(df.head())

5-node DAG with 7 edges
Sample size: 1000
Nodes: ['Z', 'X', 'M', 'Y', 'W']
Edges: [('Z', 'X'), ('Z', 'Y'), ('X', 'M'), ('X', 'Y'), ('M', 'Y'), ('X', 'W'), ('Y', 'W')]
True direct effect X -> Y: 0.4
True total effect X -> Y: 0.340

Head of dataset:
          Z         X         M         Y         W
0  0.304717  0.093076 -0.424028  1.462474  1.030979
1 -1.039984 -1.249279 -1.040661  0.084118  0.312638
2  0.750451 -0.039247  0.422236  2.091117  1.299256
3  0.940565  1.104193  0.583112 -0.940609  2.320624
4 -1.951035 -0.972524 -1.696549 -0.900371  0.493340


In [2]:
from IPython.display import Markdown, display
display(Markdown(
    '## Step 1: Marginal correlations\n\n'
    '先看 X, Y, Z, M, W 两两之间的边际相关系数。'
    'X 与 Y 显著正相关（同时受 Z 与 M 影响）；'
    'W 与 X、W 与 Y 也正相关（W 是 collider，X 与 Y 共同决定 W）。'
))

corr = df.corr().round(3)
print('Marginal correlation matrix:')
print(corr)

## Step 1: Marginal correlations

先看 X, Y, Z, M, W 两两之间的边际相关系数。X 与 Y 显著正相关（同时受 Z 与 M 影响）；W 与 X、W 与 Y 也正相关（W 是 collider，X 与 Y 共同决定 W）。

Marginal correlation matrix:
       Z      X      M      Y      W
Z  1.000  0.432  0.165  0.406  0.339
X  0.432  1.000  0.307  0.436  0.585
M  0.165  0.307  1.000  0.005  0.151
Y  0.406  0.436  0.005  1.000  0.587
W  0.339  0.585  0.151  0.587  1.000


In [3]:
from itertools import combinations

def partial_corr(df, x, y, given):
    """Pearson partial correlation of x and y given a list/set `given`."""
    if not given:
        return df[[x, y]].corr().iloc[0, 1]
    from numpy.linalg import lstsq
    Z = df[list(given)].values
    Z = np.column_stack([np.ones(len(Z)), Z])
    res_x = df[x].values - Z @ lstsq(Z, df[x].values, rcond=None)[0]
    res_y = df[y].values - Z @ lstsq(Z, df[y].values, rcond=None)[0]
    return float(np.corrcoef(res_x, res_y)[0, 1])

nodes = ['Z', 'X', 'M', 'Y', 'W']
print('Pairwise marginal correlations:')
for a, b in combinations(nodes, 2):
    print(f'  {a} ~ {b}: {df[a].corr(df[b]):+.3f}')

Pairwise marginal correlations:
  Z ~ X: +0.432
  Z ~ M: +0.165
  Z ~ Y: +0.406
  Z ~ W: +0.339
  X ~ M: +0.307
  X ~ Y: +0.436
  X ~ W: +0.585
  M ~ Y: +0.005
  M ~ W: +0.151
  Y ~ W: +0.587


In [4]:
display(Markdown(
    '## Step 2: d-separation 三种基本结构\n\n'
    'd-separation 定理刻画的是**路径**是否被条件集合阻断。先在孤立的 3 节点结构上演示定理，'
    '再回到 5-node DAG 看多路径叠加效果。'
))

# Isolated 3-node structure demos: each built with FRESH data so no extra edges.
rng = np.random.default_rng(2024)

# (1) Pure chain: Z -> X -> Y, no other edges
Z_chain = rng.normal(0, 1, 1000)
X_chain = 0.5 * Z_chain + rng.normal(0, 1, 1000)
Y_chain = 0.5 * X_chain + rng.normal(0, 1, 1000)
chain_df = pd.DataFrame({'Z': Z_chain, 'X': X_chain, 'Y': Y_chain})

# (2) Pure fork: Z -> X, Z -> Y (Z is confounder)
Z_fork = rng.normal(0, 1, 1000)
X_fork = 0.5 * Z_fork + rng.normal(0, 1, 1000)
Y_fork = 0.5 * Z_fork + rng.normal(0, 1, 1000)
fork_df = pd.DataFrame({'Z': Z_fork, 'X': X_fork, 'Y': Y_fork})

# (3) Pure collider: X -> W <- Y, no other edges
X_col = rng.normal(0, 1, 1000)
Y_col = rng.normal(0, 1, 1000)
W_col = 0.5 * X_col + 0.5 * Y_col + rng.normal(0, 1, 1000)
col_df = pd.DataFrame({'X': X_col, 'Y': Y_col, 'W': W_col})

print('=== Pure chain Z -> X -> Y (test: Z _||_ Y | X) ===')
print(f'  corr(Z, Y)            = {chain_df["Z"].corr(chain_df["Y"]):+.3f}')
print(f'  partial_corr(Z, Y|X)  = {partial_corr(chain_df, "Z", "Y", ["X"]):+.3f}')
print('  -> controlling the middle X blocks the Z->X->Y path: Z and Y become independent')
print()

print('=== Pure fork Z -> {X, Y} (test: X _||_ Y | Z) ===')
print(f'  corr(X, Y)            = {fork_df["X"].corr(fork_df["Y"]):+.3f}')
print(f'  partial_corr(X, Y|Z)  = {partial_corr(fork_df, "X", "Y", ["Z"]):+.3f}')
print('  -> controlling the common cause Z removes confounding: X and Y become independent')
print()

print('=== Pure collider X -> W <- Y (test: X _||_ Y | W opens path) ===')
print(f'  corr(X, Y)            = {col_df["X"].corr(col_df["Y"]):+.3f}  (marginally independent)')
print(f'  partial_corr(X, Y|W)  = {partial_corr(col_df, "X", "Y", ["W"]):+.3f}  (CONTROL OPENS PATH)')
print('  -> conditioning on collider W induces spurious X-Y correlation')
print()

print('=== 5-node DAG (multi-path) ===')
print(f'  corr(X, Y)            = {partial_corr(df, "X", "Y", []):+.3f}')
print(f'  partial_corr(X, Y|Z)  = {partial_corr(df, "X", "Y", ["Z"]):+.3f}')
print(f'  partial_corr(X, Y|W)  = {partial_corr(df, "X", "Y", ["W"]):+.3f}')
print('  -> controlling Z removes Z\'s confounding but X->Y (direct) and X->M->Y remain')
print('  -> controlling W (collider) still induces spurious X-Y correlation')

## Step 2: d-separation 三种基本结构

d-separation 定理刻画的是**路径**是否被条件集合阻断。先在孤立的 3 节点结构上演示定理，再回到 5-node DAG 看多路径叠加效果。

=== Pure chain Z -> X -> Y (test: Z _||_ Y | X) ===
  corr(Z, Y)            = +0.228
  partial_corr(Z, Y|X)  = -0.010
  -> controlling the middle X blocks the Z->X->Y path: Z and Y become independent

=== Pure fork Z -> {X, Y} (test: X _||_ Y | Z) ===
  corr(X, Y)            = +0.234
  partial_corr(X, Y|Z)  = +0.024
  -> controlling the common cause Z removes confounding: X and Y become independent

=== Pure collider X -> W <- Y (test: X _||_ Y | W opens path) ===
  corr(X, Y)            = -0.003  (marginally independent)
  partial_corr(X, Y|W)  = -0.214  (CONTROL OPENS PATH)
  -> conditioning on collider W induces spurious X-Y correlation

=== 5-node DAG (multi-path) ===
  corr(X, Y)            = +0.436
  partial_corr(X, Y|Z)  = +0.316
  partial_corr(X, Y|W)  = +0.142
  -> controlling Z removes Z's confounding but X->Y (direct) and X->M->Y remain
  -> controlling W (collider) still induces spurious X-Y correlation


In [5]:
def ols_coef(df, y_col, x_cols):
    """Plain-OLS coefficient on the FIRST regressor (treatment X = x_cols[0]).

    Other regressors (Z, M, W) are included in the regression but their
    coefficients are discarded. This is the coefficient on the **cause** of
    interest, not the last covariate in the formula.
    """
    X = df[x_cols].values
    X = np.column_stack([np.ones(len(X)), X])
    y = df[y_col].values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return float(beta[1])  # intercept=beta[0]; treatment X=beta[1]

direct = data['coeffs']['X->Y']
total = direct + data['coeffs']['X->M'] * data['coeffs']['M->Y']
print(f'True direct effect X -> Y = {direct:.2f}')
print(f'True total effect  X -> Y = {total:.3f}  (= direct + X->M * M->Y)')
print()

# Naive regression (no adjustment): confounded by Z + the indirect X->M->Y path
naive_coef = ols_coef(df, 'Y', ['X'])
print(f'Naive Y ~ X:          b_X = {naive_coef:+.3f}  (true total = {total:.2f})')
print('  -> b_X inflated by Z (confounder) and the X->M->Y indirect path')
print()

# Backdoor adjustment: control {Z} only (Pearl 1995 criterion)
bd_coef = ols_coef(df, 'Y', ['X', 'Z'])
print(f'Backdoor Y ~ X + Z:   b_X = {bd_coef:+.3f}  (true total = {total:.2f})')
print('  -> recovers the TOTAL causal effect of X on Y (direct + indirect via M)')
print()

# Wrong: also control M (mediator) -> over-adjustment closes X->M->Y path
wrong_coef = ols_coef(df, 'Y', ['X', 'Z', 'M'])
print(f'Wrong Y ~ X + Z + M:  b_X = {wrong_coef:+.3f}  (true direct = {direct:.2f})')
print('  -> over-adjustment closes X->M->Y, recovers only DIRECT effect (not total)')
print()

# Wrong: control W (collider) -> induces spurious correlation, biases b_X toward 0
collider_coef = ols_coef(df, 'Y', ['X', 'Z', 'W'])
print(f'Wrong Y ~ X + Z + W:  b_X = {collider_coef:+.3f}  (true total = {total:.2f})')
print('  -> collider W opens spurious X-Y path, biases b_X toward 0')

True direct effect X -> Y = 0.40
True total effect  X -> Y = 0.340  (= direct + X->M * M->Y)

Naive Y ~ X:          b_X = +0.442  (true total = 0.34)
  -> b_X inflated by Z (confounder) and the X->M->Y indirect path

Backdoor Y ~ X + Z:   b_X = +0.325  (true total = 0.34)
  -> recovers the TOTAL causal effect of X on Y (direct + indirect via M)

Wrong Y ~ X + Z + M:  b_X = +0.370  (true direct = 0.40)
  -> over-adjustment closes X->M->Y, recovers only DIRECT effect (not total)

Wrong Y ~ X + Z + W:  b_X = +0.065  (true total = 0.34)
  -> collider W opens spurious X-Y path, biases b_X toward 0


In [6]:
coefs = [naive_coef, bd_coef, wrong_coef, collider_coef]

## 实战小结

**纯结构 d-separation（3 节点演示）**：
- Chain Z → X → Y：marginal corr(Z, Y) = +0.23，partial_corr(Z, Y|X) = -0.01（链被关）
- Fork Z → {X, Y}：marginal corr(X, Y) = +0.23，partial_corr(X, Y|Z) = +0.02（叉被关）
- Collider X → W ← Y：marginal corr(X, Y) ≈ 0（天然独立），控制 W 后 = -0.21（对撞打开伪路径）

**5-node DAG 多路径演示**：
- Marginal corr(X, Y) = +0.44：Z (confounder) + X→Y 直接 + X→M→Y 间接叠加
- partial_corr(X, Y | Z) = +0.32：控制 Z 后只剩 X→Y 直接 + X→M→Y 间接（≈ 0.34 总因果效应）
- partial_corr(X, Y | W) = +0.14：控制 collider W 引入伪路径

**Backdoor 调整（5-node DAG，全部是 b_X）**：
- Naive Y ~ X：b_X ≈ +0.44，被 Z（confounder）与 X→M→Y 间接路径双重抬升
- **Backdoor Y ~ X + Z：b_X ≈ +0.32，恢复到 X 对 Y 的总因果效应（≈ 0.34）**
- 错误调整 M（mediator）：b_X ≈ +0.37，只恢复直接效应（≈ 0.40），丢失 X→M→Y 间接路径
- 错误调整 W（collider）：b_X ≈ +0.07，被 X→W←Y 伪路径拖向 0

局限：
- 真实业务里 DAG 节点画错任何一个，整个识别都失效（领域知识决定）
- 高维 DAG（20+ 节点）需要 Shpitser et al. 2010 的 ID 算法自动找调整集
- 反馈循环系统（动态 DAG）需要 do-calculus + 时间切片，本文未涉及